# Taming the N×N Bottleneck: An Interactive Tour of Efficient Transformers

Self-attention is powerful, but it has one expensive habit: it compares **every** token with **every** other token, forming an **N×N attention matrix**. Both the compute and the memory of that matrix grow as **O(N²)** in the sequence length `N`. When `N` is large — think a flattened image with `N = 256 × 256` — that single matrix dominates the entire model's cost. **Every 'efficient transformer' is an answer to this one bottleneck.**

This notebook builds the story up in four measured steps:

1. **WHY attention is expensive** — render the N×N matrix and watch runtime and memory blow up on a quadratic curve as `N` grows.
2. **Skip calculations with human knowledge** — local / stride / global *sparse masks*, and how few entries they actually compute.
3. **Do we need the full matrix?** — the *low-rank* view: compress `N` keys to `K` representative keys (Linformer) for linear cost.
4. **The math** — approximate softmax with a feature map φ so matrix-product *associativity* reorders the cost from `(d+d')N²` down to `2d'dN` (Linear Transformer, Performer).

**Demo philosophy:** everything runs on small *synthetic* sequences with plain PyTorch / NumPy — no pretrained models, no datasets, no GPU required. The whole notebook executes end-to-end in well under a minute, and **every efficiency claim is something you measure, not something you take on faith.**

In [ ]:
# All dependencies (torch, numpy, matplotlib, ipywidgets) are preinstalled on Google Colab.
# If ipywidgets is somehow missing, uncomment the next line:
# !pip install -q ipywidgets

import math
import time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, Dropdown

# Enable inline plotting, but stay import-safe outside IPython.
if 'get_ipython' in globals() and get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')

# Reproducibility: fix all RNG seeds.
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection. All demos are tiny synthetic sequences and run fine on CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Consistent plot styling for every heatmap / line plot below.
plt.rcParams['figure.figsize'] = (5, 4)
plt.rcParams['image.cmap'] = 'viridis'
plt.rcParams['axes.grid'] = False

# Sanity checks.
assert device.type in ('cpu', 'cuda')
assert SEED == 0 and isinstance(SEED, int)

print(f'Setup OK | torch {torch.__version__} | device={device} | seed={SEED}')

In [ ]:
# Shared attention toolkit -- the math is written exactly once here and reused everywhere.

D_HEAD = 32  # per-token feature dimension (one attention head)


def make_qkv(N, d=D_HEAD, scale=1.0):
    '''Random synthetic Q, K, V for one length-N attention head, each (N, d) on device.'''
    Q = torch.randn(N, d, device=device) * scale
    K = torch.randn(N, d, device=device) * scale
    V = torch.randn(N, d, device=device) * scale
    return Q, K, V


def softmax_attention(Q, K, V, mask=None):
    '''Reference full attention. Returns (O, A) where A is the N x N attention matrix.

    mask (optional): boolean (N, N) tensor, True = keep. Every mask used in this
    notebook keeps at least the diagonal True, so softmax never reduces over an
    all -inf row (which would produce NaN).
    '''
    d = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    A = torch.softmax(scores, dim=-1)
    O = A @ V
    return O, A


def plot_heatmap(M, title, ax=None):
    '''Render a matrix (torch.Tensor or np.ndarray) as a heatmap.'''
    if hasattr(M, 'detach'):
        M = M.detach().cpu().numpy()
    if ax is None:
        fig, ax = plt.subplots()
    im = ax.imshow(M, aspect='auto')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
    return ax


def measure_runtime(fn, repeats=5):
    '''Median wall-clock seconds of fn() over repeats runs (after one warm-up).'''
    fn()  # warm-up: discard first-call lazy init
    times = []
    for _ in range(repeats):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


# Sanity check: attention rows are a probability distribution (sum to 1).
Q, K, V = make_qkv(8)
O, A = softmax_attention(Q, K, V)
assert torch.allclose(A.sum(dim=-1), torch.ones(8, device=device), atol=1e-5)
print(f'Toolkit OK | D_HEAD={D_HEAD} | device={device} | A.shape={tuple(A.shape)} | rows sum to 1')

## 1 · The Quadratic Bottleneck

Self-attention computes the attention matrix `A = softmax(QKᵀ / √d)` and then the output `O = A · V`.

To build `A`, **every query is compared with every key** — so `A` is an **N×N** matrix. Forming it costs `O(N²)` multiply-adds and storing it costs `O(N²)` memory. Double the sequence length and the cost roughly **quadruples**.

Attention is only one module inside a much larger network, but it **dominates the cost when `N` is large**. The lecture's motivating case is image processing: flatten a `256 × 256` image and you already have `N = 65,536` tokens — an attention matrix with over four *billion* entries. That is exactly why efficient-attention research exists.

In the next cell we literally **render the N×N matrix** and **measure** how runtime and memory grow as `N` increases, plotting them against a quadratic reference. The 'quadratic' claim becomes a curve you can see, not a word you take on faith.

In [ ]:
# --- 1. The dense N x N attention matrix (every token attends to every token) ---
N0 = 64
Q, K, V = make_qkv(N0)
O, A = softmax_attention(Q, K, V)
fig, ax = plt.subplots()
plot_heatmap(A, f'Dense N x N attention (N={N0})', ax=ax)
plt.show()
attention_matrix_figure = fig

# --- 2. Measure runtime and entry count as N grows ---
seq_lengths = [64, 128, 256, 512, 1024]
runtimes = []
entry_counts = []
for N in seq_lengths:
    Q, K, V = make_qkv(N)
    # Bind Q, K, V as default args to avoid Python late-binding in the closure.
    t = measure_runtime(lambda Q=Q, K=K, V=V: softmax_attention(Q, K, V))
    runtimes.append(t)
    entry_counts.append(N * N)

# --- 3. Plot measured runtime vs a quadratic reference, and entry count = N^2 ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(seq_lengths, runtimes, 'o-', label='measured')
ref = [runtimes[0] * (N / seq_lengths[0]) ** 2 for N in seq_lengths]
axes[0].plot(seq_lengths, ref, '--', label='N^2 reference')
axes[0].set_xscale('log', base=2)
axes[0].set_yscale('log')
axes[0].set_xlabel('sequence length N')
axes[0].set_ylabel('runtime (s)')
axes[0].set_title('runtime vs N')
axes[0].legend()
axes[1].plot(seq_lengths, entry_counts, 's-')
axes[1].set_xscale('log', base=2)
axes[1].set_yscale('log')
axes[1].set_xlabel('sequence length N')
axes[1].set_ylabel('attention entries')
axes[1].set_title('attention entries = N^2 (memory proxy)')
plt.tight_layout()
plt.show()
cost_vs_n_figure = fig

# --- 4. Print the doubling factor: quadratic cost roughly 4x per doubling of N ---
assert len(runtimes) == len(seq_lengths) == len(entry_counts)
assert entry_counts == [n * n for n in seq_lengths]
for i in range(len(seq_lengths) - 1):
    assert runtimes[i] > 0
    print(f'N {seq_lengths[i]}->{seq_lengths[i+1]}: runtime x{runtimes[i+1]/runtimes[i]:.2f} (quadratic~4x)')

In [ ]:
# Interactive bottleneck explorer: drag N and watch the matrix -- and its cost -- grow.

def explore_bottleneck(N):
    Q, K, V = make_qkv(N)
    t = measure_runtime(lambda: softmax_attention(Q, K, V))
    O, A = softmax_attention(Q, K, V)
    fig, ax = plt.subplots()
    plot_heatmap(A, f'Attention matrix (N={N})', ax=ax)
    plt.show()
    print(f'matrix size = {N} x {N} = {N*N:,} entries | runtime ~ {t*1e3:.2f} ms')


# Quick direct check before wiring the widget (prints 4096 entries, runs without error).
explore_bottleneck(64)

# Slider capped at 512 so the quadratic branch can never hang the kernel.
# interact() calls explore_bottleneck once with value=128, so a static result
# renders on first execution even if the widget JS fails to initialize.
bottleneck_widget = interact(
    explore_bottleneck,
    N=IntSlider(min=16, max=512, step=16, value=128, continuous_update=False),
)

## 2 · Skip Calculations with Human Knowledge

The most intuitive way to cut the N×N cost is to decide **in advance** that many attention entries are unnecessary and simply zero them out. Three building-block patterns from the lecture:

- **Local / truncated attention** — compute weights only inside a nearby window and set the rest to 0, exactly like a CNN's receptive field.
- **Stride / dilated attention** — attend to every `k`-th token, widening the field of view cheaply.
- **Global attention** — add a few special tokens that attend to *and* are attended by **every** token, while ordinary tokens stay local.

Real models **combine** these: **Longformer** = sliding window + dilation + a few global tokens; **Big Bird** = random + window + global.

In the next cell each pattern becomes a **boolean mask** we can render as a heatmap and whose **computed-entry count** we can measure against the dense baseline — so you see precisely how much work each pattern saves.

In [ ]:
# Three human-knowledge sparsity patterns, expressed as boolean attention masks.

def local_mask(N, window):
    '''True within +/- window of the diagonal (CNN-like local neighborhood).'''
    window = max(0, min(window, N - 1))
    idx = torch.arange(N)
    diff = (idx[:, None] - idx[None, :]).abs()
    return (diff <= window).to(device)


def stride_mask(N, stride):
    '''True on every stride-th column, plus the diagonal so no row is empty.'''
    stride = max(1, min(stride, N))
    cols = torch.arange(N)
    base = (cols[None, :] % stride == 0).expand(N, N)
    return (base | torch.eye(N, dtype=torch.bool)).to(device)


def global_mask(N, num_global):
    '''First num_global tokens attend to / are attended by all; plus diagonal.'''
    num_global = max(0, min(num_global, N))
    m = torch.zeros(N, N, dtype=torch.bool)
    m[:, :num_global] = True   # everyone attends to the global tokens
    m[:num_global, :] = True   # global tokens attend to everyone
    m |= torch.eye(N, dtype=torch.bool)
    return m.to(device)


def mask_density(mask):
    '''Fraction of entries actually computed (True) in [0, 1].'''
    return mask.float().mean().item()


# Build the four patterns at a fixed N.
N1 = 64
local = local_mask(N1, 4)
stride = stride_mask(N1, 8)
global_ = global_mask(N1, 4)
combined = local_mask(N1, 4) | global_mask(N1, 4)  # Longformer-style: sliding + global

# Render all four as heatmaps with their density annotated.
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
panels = [('local (window=4)', local), ('stride (stride=8)', stride),
          ('global (num_global=4)', global_), ('local+global', combined)]
for (name, mask), ax in zip(panels, axes.ravel()):
    plot_heatmap(mask, f'{name}  density={mask_density(mask)*100:.1f}%', ax=ax)
plt.tight_layout()
plt.show()
sparse_masks_figure = fig

# Apply one pattern (local) and compare against dense attention.
Q, K, V = make_qkv(N1)
O_dense, A_dense = softmax_attention(Q, K, V)
O_local, A_local = softmax_attention(Q, K, V, mask=local)
rel_diff = (torch.norm(O_local - O_dense) / torch.norm(O_dense)).item()

ones = torch.ones(N1, device=device)
assert torch.allclose(A_local.sum(-1), ones, atol=1e-5)  # masked attention still normalized
print(f'local-vs-dense output relative difference = {rel_diff:.3f}')
print(f'dense keeps {N1*N1}; local {int(local.sum())}; stride {int(stride.sum())}; global {int(global_.sum())}')

In [ ]:
# Interactive sparse-attention explorer: pick a pattern, sweep its parameter,
# watch the sparsity-vs-fidelity trade-off. Q,K,V are fixed so only the mask changes.

N2 = 64
Q2, K2, V2 = make_qkv(N2)


def explore_sparse(pattern, window, stride, num_global):
    # Sliders not relevant to the chosen pattern simply have no effect.
    if pattern == 'local':
        mask = local_mask(N2, window)
    elif pattern == 'stride':
        mask = stride_mask(N2, stride)
    elif pattern == 'global':
        mask = global_mask(N2, num_global)
    else:  # 'local+global'
        mask = local_mask(N2, window) | global_mask(N2, num_global)

    d = mask_density(mask)
    speedup = 1.0 / max(d, 1e-9)
    O_m, A_m = softmax_attention(Q2, K2, V2, mask=mask)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    plot_heatmap(mask, f'{pattern} mask  density={d*100:.1f}%', ax=axes[0])
    plot_heatmap(A_m, 'masked attention', ax=axes[1])
    plt.tight_layout()
    plt.show()
    print(f'computes {d*100:.1f}% of entries -> ~{speedup:.1f}x fewer attention entries vs dense')


# Direct check before wiring (renders the default local-pattern view).
explore_sparse('local', 4, 8, 4)

sparse_widget = interact(
    explore_sparse,
    pattern=Dropdown(options=['local', 'stride', 'global', 'local+global'], value='local'),
    window=IntSlider(min=1, max=N2 // 2, step=1, value=4, continuous_update=False),
    stride=IntSlider(min=1, max=N2, step=1, value=8, continuous_update=False),
    num_global=IntSlider(min=0, max=N2 // 4, step=1, value=4, continuous_update=False),
)

## 3 · Do We Need the *Full* Attention Matrix?

Look at a real attention matrix and you find it has **many redundant columns** — it is effectively **low rank**. If the matrix barely changes when you drop most of its directions, why pay for all `N` keys?

**Linformer** acts on this: it learns an `N×K` linear projection that **compresses the `N` keys (and values) down to `K << N` representative ones** *before* attention. The attention matrix becomes `N×K` instead of `N×N`, making the cost **linear in `N`**. (Compressed Attention does the same with a convolution.)

One subtlety the lecture stresses: we reduce the number of **keys / values, not queries**. Reducing queries would shrink the output sequence length — we still want one output vector per input token.

In the next cell we **read off the singular-value spectrum** (the visible 'low rank') and compare **Linformer-style compressed attention** against full attention as we vary `K`.

In [ ]:
# The low-rank view of attention + a Linformer-style compression.

N3 = 128
Q, K, V = make_qkv(N3)
O_full, A = softmax_attention(Q, K, V)

# --- 1. Singular-value spectrum: energy concentrates in a few components ---
assert torch.isfinite(A).all()
s = torch.linalg.svdvals(A)          # descending singular values, shape (N3,)
s_np = s.detach().cpu().numpy()
fig, ax = plt.subplots()
ax.plot(s_np, 'o-')
ax.set_yscale('log')
ax.set_title('Singular values of attention matrix (fast decay = low rank)')
ax.set_xlabel('index')
ax.set_ylabel('singular value (log)')
plt.show()
singular_value_figure = fig
assert torch.all(s[:-1] >= s[1:] - 1e-6)  # non-increasing
top10_energy = (s[:10].sum() / s.sum()).item()
print(f'top-10 singular values hold {top10_energy*100:.1f}% of the spectral energy')


def linformer_attention(Q, K, V, k):
    '''Compress N keys/values to k representative ones via a FIXED random projection.

    Returns (O, A_lin) with A_lin of shape (N, k); cost is N x k instead of N x N.
    '''
    N, d = K.shape
    k = max(1, min(k, N))
    E = torch.randn(N, k, device=device) / math.sqrt(N)  # fixed (not learned) projection
    K_proj = E.transpose(0, 1) @ K   # (k, d)
    V_proj = E.transpose(0, 1) @ V   # (k, d)
    scores = (Q @ K_proj.transpose(0, 1)) / math.sqrt(d)  # (N, k)
    A_lin = torch.softmax(scores, dim=-1)                 # (N, k)
    O = A_lin @ V_proj                                    # (N, d)
    return O, A_lin


# --- 2. Compare full N x N attention against the compact N x k Linformer matrix ---
O_lin, A_lin = linformer_attention(Q, K, V, 16)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_heatmap(A, f'full N x N (N={N3})', ax=axes[0])
plot_heatmap(A_lin, 'Linformer N x k (k=16)', ax=axes[1])
plt.tight_layout()
plt.show()
lowrank_compare_figure = fig

# --- 3. How close is the compressed output? ---
ones = torch.ones(N3, device=device)
assert torch.allclose(A_lin.sum(-1), ones, atol=1e-5)
cos = F.cosine_similarity(O_full.flatten(), O_lin.flatten(), dim=0).item()
rel_err = (torch.norm(O_full - O_lin) / torch.norm(O_full)).item()
print(f'k=16: cosine={cos:.3f}, rel_err={rel_err:.3f}, cost N x k={N3*16} vs N x N={N3*N3}')

In [ ]:
# Interactive low-rank explorer: slide k and watch the accuracy-vs-cost trade-off.
# Rebuild the baseline so this cell is self-sufficient even on a partial run.
N3 = 128
Q, K, V = make_qkv(N3)
O_full, _ = softmax_attention(Q, K, V)


def explore_lowrank(k):
    O_lin, A_lin = linformer_attention(Q, K, V, k)
    cos = F.cosine_similarity(O_full.flatten(), O_lin.flatten(), dim=0).item()
    rel_err = (torch.norm(O_full - O_lin) / torch.norm(O_full)).item()
    cost_ratio = (N3 * k) / (N3 * N3)
    fig, ax = plt.subplots()
    plot_heatmap(A_lin, f'Linformer attention N x k (k={k})', ax=ax)
    plt.show()
    print(f'k={k}: cosine={cos:.3f} | rel_err={rel_err:.3f} | cost = N x k / N x N = {cost_ratio:.3f}')


# Direct check before wiring (renders the default k=16 view).
explore_lowrank(16)

lowrank_widget = interact(
    explore_lowrank,
    k=IntSlider(min=1, max=128, step=1, value=16, continuous_update=False),
)

## 4 · The Mathematical Heart: Linear Attention by Reordering

Ignore the softmax for a moment. The output is a product of three matrices, and by **associativity** `(A·C)·P = A·(C·P)` the **order** of multiplication does not change the *result* — but it dramatically changes the *cost*.

Forming the big `N×N` term first costs `O((d+d')N²)`. Instead, compute the small `d'×d` summary `Σⱼ φ(kⱼ)·vⱼᵀ` **once** and reuse it across all queries: that costs only `O(2 d' d N)` — **linear in `N`**. (Same reason a 10⁶-element job can collapse to ~10³ when you multiply in the smart order.)

To put softmax back, approximate `exp(q·k) ≈ φ(q)·φ(k)` with a **kernel feature map** φ. Then a single shared sum over keys is reused for every query — *do not compute it again*. This is the **Linear Transformer** (φ = elu + 1) and the **Performer** (random features).

In the next cell we **verify** the two multiplication orders give an **identical** output (`torch.allclose`), then **measure** the quadratic-vs-linear runtime gap widening as `N` grows.

In [ ]:
# The mathematical heart: associativity turns O(N^2) attention into O(N) attention.

def feature_map(x):
    '''Linear Transformer kernel phi(x) = elu(x) + 1 (strictly positive).'''
    return F.elu(x) + 1.0


def linear_attention(Q, K, V):
    '''Linear-cost order: build a shared KV summary once, reuse it for every query.'''
    Qf = feature_map(Q)                     # (N, dp)
    Kf = feature_map(K)                     # (N, dp)
    S = Kf.transpose(0, 1) @ V              # (dp, d) shared KV summary
    z = Kf.sum(dim=0)                       # (dp,) normalizer = Kf^T @ ones
    num = Qf @ S                            # (N, d)
    den = (Qf @ z).unsqueeze(-1)            # (N, 1)
    return num / (den + 1e-6)


# --- Associativity proof: the SAME quantity computed the expensive N x N way ---
Q, K, V = make_qkv(256)
Qf, Kf = feature_map(Q), feature_map(K)
A_unnorm = Qf @ Kf.transpose(0, 1)          # (N, N) -- the expensive order
num2 = A_unnorm @ V
den2 = A_unnorm.sum(dim=1, keepdim=True)
O_slow = num2 / (den2 + 1e-6)
O_fast = linear_attention(Q, K, V)
order_equiv_check = torch.allclose(O_fast, O_slow, atol=1e-5)
assert order_equiv_check, f'reordering must not change result; max diff={(O_fast-O_slow).abs().max().item()}'
print(f'associativity check: two multiplication orders agree -> {order_equiv_check}')

# --- Linear attention only APPROXIMATES softmax (never asserted equal) ---
O_full, _ = softmax_attention(Q, K, V)
cos = F.cosine_similarity(O_full.flatten(), O_fast.flatten(), dim=0).item()
print(f'linear vs softmax cosine ~ {cos:.3f} (approximation, not exact)')

# --- Crossover sweep: quadratic softmax vs linear attention as N grows ---
lin_seq = [128, 256, 512, 1024, 2048]
quad_t, lin_t = [], []
for N in lin_seq:
    Q, K, V = make_qkv(N)
    quad_t.append(measure_runtime(lambda Q=Q, K=K, V=V: softmax_attention(Q, K, V)))
    lin_t.append(measure_runtime(lambda Q=Q, K=K, V=V: linear_attention(Q, K, V)))

fig, ax = plt.subplots()
ax.plot(lin_seq, quad_t, 'o-', label='softmax O(N^2)')
ax.plot(lin_seq, lin_t, 's-', label='linear O(N)')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('sequence length N')
ax.set_ylabel('runtime (s)')
ax.set_title('quadratic vs linear attention')
ax.legend()
plt.show()
linear_vs_quadratic_figure = fig
print(f'at N={lin_seq[-1]}: softmax {quad_t[-1]*1e3:.2f} ms vs linear {lin_t[-1]*1e3:.2f} ms -> {quad_t[-1]/max(lin_t[-1],1e-9):.1f}x')

In [ ]:
# Interactive linear-attention explorer: push N up and watch linear pull ahead.

def explore_linear(N):
    Q, K, V = make_qkv(N)
    t_quad = measure_runtime(lambda: softmax_attention(Q, K, V))
    t_lin = measure_runtime(lambda: linear_attention(Q, K, V))
    speedup = t_quad / max(t_lin, 1e-9)
    d = D_HEAD
    dp = D_HEAD                       # feature dim of the elu+1 map
    flops_quad = (d + dp) * N * N     # theoretical, not profiler-measured
    flops_lin = 2 * dp * d * N
    O_full, _ = softmax_attention(Q, K, V)
    O_lin = linear_attention(Q, K, V)
    cos = F.cosine_similarity(O_full.flatten(), O_lin.flatten(), dim=0).item()
    print(f'N={N}: softmax {t_quad*1e3:.2f} ms vs linear {t_lin*1e3:.2f} ms -> {speedup:.1f}x '
          f'| FLOPs(theory) quad={flops_quad:,} lin={flops_lin:,} | agreement cos={cos:.3f}')
    fig, ax = plt.subplots()
    ax.bar(['softmax O(N^2)', 'linear O(N)'], [t_quad * 1e3, t_lin * 1e3])
    ax.set_ylabel('ms')
    ax.set_title(f'runtime at N={N}')
    plt.show()


# Direct check before wiring (renders the default N=512 comparison).
explore_linear(512)

# N capped at 2048 so the quadratic branch cannot hang the kernel.
linear_widget = interact(
    explore_linear,
    N=IntSlider(min=128, max=2048, step=128, value=512, continuous_update=False),
)

## Wrap-Up: One Bottleneck, Four Answers

We followed a single spine — the **O(N²)** cost of the N×N attention matrix — and saw four progressively more powerful answers:

1. **The bottleneck** — the N×N matrix; runtime and memory grow quadratically with `N`. *(Objective 1)*
2. **Sparse masks** — skip entries using human knowledge: local / stride / global patterns (Longformer, Big Bird). *(Objective 2)*
3. **Low rank** — the matrix is redundant, so compress `N` keys to `K` representative keys for linear cost (Linformer). *(Objective 3)*
4. **Linear attention** — a feature map φ plus matrix-product associativity gives truly linear attention (Linear Transformer, Performer). *(Objective 4)*

Every efficiency claim here was **measured on the runtime curves**, not asserted.

**Intentionally left out** to keep this notebook focused and reliably executable — strong candidates for a follow-up notebook:

- **Clustering-based attention** — Reformer (LSH), Routing Transformer.
- **Learnable sparsity patterns** — Sinkhorn Sorting Network.
- **Attention weights as free parameters** — Synthesizer.
- **Attention-free token mixing** — FNet (Fourier mixing), MLP-Mixer, gMLP.